# King County House Prices Project

Your task is to **predict the sale price of a house** in King County with a regression model, and to **tune** the model with cross-validation. In the project you combine the methods you already know into one complete analysis.

The task below follows the steps of a machine learning project, in a single pass:

![The project in one pass, from top to bottom: 1. goal and metric, 2. get the data, 3. a train-test split into training data and locked test data. Steps 4 and 5 (explore, baselines) use only the training data. Steps 6 and 7 form the model selection loop: feature engineering, set hyperparameters, train model(s) and calculate the cross-validation score, then repeat. In step 8, the test data is unlocked and the models are scored on the test set](assets/king_county_project_workflow.png)

# Data Description

King County covers Seattle and its surroundings. The data ships with this repository as **two tables**:

- `data/king_county_house_sales.csv`: one row per sale, with the date and the price
- `data/king_county_house_details.csv`: one row per house, with its physical attributes

`01_loading_the_data.ipynb` guides you through combining them into one table.

| Column | Table | Description |
| :-- | :-- | :-- |
| `date` | sales | date the house was sold |
| `price` | sales | sale price in dollars, the prediction target |
| `house_id` | sales | identifier of the house, matches `id` in the details table |
| `id` | sales | identifier of the sale |
| `id` | details | unique identifier for a house |
| `bedrooms` | details | number of bedrooms |
| `bathrooms` | details | number of bathrooms |
| `sqft_living` | details | square footage of the home |
| `sqft_lot` | details | square footage of the lot |
| `floors` | details | total floors (levels) in the house |
| `waterfront` | details | whether the house has a view to a waterfront |
| `view` | details | quality of the view |
| `condition` | details | overall condition of the house |
| `grade` | details | overall grade given to the housing unit, based on the King County grading system |
| `sqft_above` | details | square footage of the house apart from the basement |
| `sqft_basement` | details | square footage of the basement |
| `yr_built` | details | year the house was built |
| `yr_renovated` | details | year the house was renovated |
| `zipcode` | details | zip code |
| `lat` | details | latitude coordinate |
| `long` | details | longitude coordinate |
| `sqft_living15` | details | square footage of the interior living space of the nearest 15 neighbours |
| `sqft_lot15` | details | square footage of the land lots of the nearest 15 neighbours |

## Task

1. **Set your goal.** Decide which metric you use to judge the model, and explain why.
2. **Get the data.** Complete `01_loading_the_data.ipynb` first: it combines the two tables and saves the result as `data/eda.csv`, which you load here.
3. **Split the data into a training and a test set.** Use `train_test_split` once. Choose the size of the test set with `test_size`, and fix the `random_state`, so that you get the same split every time you run the notebook. Keep the features and the target together: the result should be two DataFrames, `df_train` and `df_test`, that both still contain the `price` column. You do not need a separate validation set: in step 7, cross-validation on the training set takes its place. Do not look at `df_test` until step 8.
4. **Explore the training set.** Which columns are related to the price, and which one most strongly? Which columns are skewed, which values are missing, and which features contain the same information?

> [!IMPORTANT]
> **Keep a record of every change you make to the training data by hand.** In step 8 you apply exactly the same changes to the test set. This covers, for example, a new column such as the age of the house at sale, or dropping the identifiers `id` and `house_id` and the `date` column. Transformations inside your `Pipeline` (imputing, scaling, encoding) are applied to the test set automatically, with the values learned on the training data.

5. **Separate the features from the target, and build two baselines.** Split `df_train` into `X_train` and `y_train`; the test set stays untouched until step 8. Then build two simple models that your models have to beat:
    - a **mean baseline** with [`DummyRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html), which predicts the mean price of the training set for every house, and
    - a **rule-based baseline** built from the column most strongly related to the price in step 4. Learn the values of the rule on the training data.
6. **Build a preprocessing pipeline and train a linear model.** Put every transformation into a `Pipeline` with a `ColumnTransformer`, so that it is fitted on the training data only, and train a regularised linear model, for example `Ridge`, with as many features as possible.
7. **Tune and compare models with cross-validation.** Train and tune at least two models, for example `Ridge` on the original features and on polynomial features: first a wide `RandomizedSearchCV`, then a narrow `GridSearchCV` around what it found. Choose the best model by its cross-validated score. **Save every model in its own variable** (and the values of your rule), so that you can evaluate all of them in the last step.
8. **Prepare the test set and evaluate all models at the very end.** First apply to `df_test` the changes you made by hand to the training data, and split it into `X_test` and `y_test`. Then calculate the **training and the test scores of every model** with your chosen metric, including both baselines, and collect them in **one table**: one row per model, with columns for the training and the test scores. Compare the models: which ones beat the baselines, and how large is the difference between training and test scores? Finally, **analyse the residuals** of your best model: for which houses is it wrong, and how large are the errors?

In [1]:
# Import the libraries you need for this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform
import plotly.express as px

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

2. **Get the data.** Complete `01_loading_the_data.ipynb` first: it combines the two tables and saves the result as `data/eda.csv`, which you load here.
3. **Split the data into a training and a test set.** Use `train_test_split` once. Choose the size of the test set with `test_size`, and fix the `random_state`, so that you get the same split every time you run the notebook. Keep the features and the target together: the result should be two DataFrames, `df_train` and `df_test`, that both still contain the `price` column. You do not need a separate validation set: in step 7, cross-validation on the training set takes its place. Do not look at `df_test` until step 8.
4. **Explore the training set.** Which columns are related to the price, and which one most strongly? Which columns are skewed, which values are missing, and which features contain the same information?

> [!IMPORTANT]
> **Keep a record of every change you make to the training data by hand.** In step 8 you apply exactly the same changes to the test set. This covers, for example, a new column such as the age of the house at sale, or dropping the identifiers `id` and `house_id` and the `date` column. Transformations inside your `Pipeline` (imputing, scaling, encoding) are applied to the test set automatically, with the values learned on the training data.

In [2]:
# Your code here
df = pd.read_csv("data/eda.csv")
df.head()

,date,price,house_id,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.0,7129300520,1,3.0,1.00,1180.0,5650.0,1.0,NaN,...,7,1180.0,0.0,1955,0.0,98178,47.5112,-122.257,1340.0,5650.0
1,2014-12-09,538000.0,6414100192,2,3.0,2.25,2570.0,7242.0,2.0,0.0,...,7,2170.0,400.0,1951,1991.0,98125,47.7210,-122.319,1690.0,7639.0
2,2015-02-25,180000.0,5631500400,3,2.0,1.00,770.0,10000.0,1.0,0.0,...,6,770.0,0.0,1933,NaN,98028,47.7379,-122.233,2720.0,8062.0
3,2014-12-09,604000.0,2487200875,4,4.0,3.00,1960.0,5000.0,1.0,0.0,...,7,1050.0,910.0,1965,0.0,98136,47.5208,-122.393,1360.0,5000.0
4,2015-02-18,510000.0,1954400510,5,3.0,2.00,1680.0,8080.0,1.0,0.0,...,8,1680.0,0.0,1987,0.0,98074,47.6168,-122.045,1800.0,7503.0


In [3]:
# Split train and test set
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((17277, 21), (4320, 21), (17277,), (4320,))

### EDA

In [4]:
# get a df with y and X for the EDA again 
df_train = pd.concat([X_train, y_train], axis=1)
df_train.head()

,date,house_id,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
6243,2014-12-22,9834201100,6244,4.0,2.00,1440.0,4855.0,2.0,0.0,0.0,...,1440.0,0.0,1972,0.0,98144,47.5717,-122.287,1300.0,4080.0,332500.0
8986,2015-05-12,7795810110,8987,3.0,1.75,1430.0,9857.0,1.0,0.0,2.0,...,1140.0,290.0,1980,0.0,98045,47.4964,-121.771,1310.0,9880.0,390000.0
9419,2014-12-29,1853081250,9420,4.0,2.75,3120.0,5000.0,2.0,0.0,0.0,...,3120.0,0.0,2010,0.0,98074,47.5940,-122.062,3200.0,5000.0,800000.0
10155,2015-03-21,2558630350,10156,4.0,2.50,2060.0,6958.0,1.0,0.0,0.0,...,1220.0,840.0,1974,NaN,98034,47.7251,-122.168,1760.0,7350.0,462000.0
10436,2014-07-01,3362401000,10437,3.0,2.00,2500.0,4080.0,1.5,0.0,0.0,...,1680.0,820.0,1922,0.0,98103,47.6813,-122.346,1550.0,3060.0,695000.0


In [5]:
df_train.view.unique()

array([ 0.,  2.,  3., nan,  4.,  1.])

In [6]:
df_train.condition.unique()

array([4, 3, 5, 2, 1])

In [7]:
df_train.grade.unique()

array([ 7,  9,  8, 11,  6, 10, 13,  5, 12,  4,  3])

In [8]:
df_train.sqft_basement.value_counts(dropna=False)

sqft_basement
0.0       10249
NaN         356
600.0       174
500.0       168
800.0       159
          ...  
1548.0        1
2600.0        1
295.0         1
1610.0        1
176.0         1
Name: count, Length: 286, dtype: int64

In [9]:
df_train.isna().sum()

date                0
house_id            0
id                  0
bedrooms            0
bathrooms           0
sqft_living         0
sqft_lot            0
floors              0
waterfront       1907
view               53
condition           0
grade               0
sqft_above          0
sqft_basement     356
yr_built            0
yr_renovated     3069
zipcode             0
lat                 0
long                0
sqft_living15       0
sqft_lot15          0
price               0
dtype: int64

In [10]:
df_train.info()

<class 'pandas.DataFrame'>
Index: 17277 entries, 6243 to 15795
Data columns (total 22 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           17277 non-null  str    
 1   house_id       17277 non-null  int64  
 2   id             17277 non-null  int64  
 3   bedrooms       17277 non-null  float64
 4   bathrooms      17277 non-null  float64
 5   sqft_living    17277 non-null  float64
 6   sqft_lot       17277 non-null  float64
 7   floors         17277 non-null  float64
 8   waterfront     15370 non-null  float64
 9   view           17224 non-null  float64
 10  condition      17277 non-null  int64  
 11  grade          17277 non-null  int64  
 12  sqft_above     17277 non-null  float64
 13  sqft_basement  16921 non-null  float64
 14  yr_built       17277 non-null  int64  
 15  yr_renovated   14208 non-null  float64
 16  zipcode        17277 non-null  int64  
 17  lat            17277 non-null  float64
 18  long           1727

In [11]:
# a lot of the data formats are not correct yet
# zipcode should be a string
# house_id and id should be a string 
# waterfront should be a bool - also has quite a few missings -> onehotencode this as a feature
# view is the quality of view, so we need to make this an ordinal feature later on - also has missings
# condition is also supposed to be ordinal, scale from 1-5 
# probably the same for grade, ordinal too
# sqft_basement has quite a few missings, maybe those dont have any basement? but theres also 0 values..

# yr_renovated definitely needs to be a int instead of a float
# yr_built and yr_renovated are intesting columns but maybe the age of the house and the years since last renovation would be the more interesting features
# -> should probably create those from scratch and then drop the columns yr_built and yr_renovated
# and then we'll see whether keeping those as a numeric feature or binning them is the better choice
# I would make this choice depending on the range of values we get. 

In [12]:
px.histogram(df_train, 
             x='price')

In [13]:
df_train.price.describe()

count    1.727700e+04
mean     5.415190e+05
std      3.689687e+05
min      7.800000e+04
25%      3.240000e+05
50%      4.500000e+05
75%      6.455000e+05
max      7.700000e+06
Name: price, dtype: float64

In [14]:
px.box(df_train, 
       y='price')

> We have a huge range of the price here, starting at a min of 78k up until 1.12M, the distribution is also quite skewed